### Importing libraries

In [32]:
# %pip install ddgs trafilatura -q

In [56]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display

### Setup

In [34]:
# api
load_dotenv()

OpenAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key= OpenAI_API_KEY)

if OpenAI_API_KEY is None:
    raise Exception ("No API Key for Open AI")
else:
    print(OpenAI_API_KEY[:8])

sk-proj-


#### Step 1: Define Tools

In [35]:
#search web tool

def search_web(query:str):
    """Seach the web using ddgs browser. Return the top 3 results"""
    #initialize a ddgs object:
    ddgs = DDGS()
    results = ddgs.text(query, max_results = 3)
    print(f'\u2705 Got results')
    #returning in json bc LLM doesnt work with dictionaries, so json converts to text
    return json.dumps(results, indent= 2)

In [36]:
#Fetching the information from the search results. Scraping the url

def fetch_url(url:str):
    """Extracting text from fetched from web result url"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f'\u2705 Got Text {len(text)} characters')
            return text
        else:
            print(f'\u274C Failed to fetch or extract text.')
    return f'Could not extract text from {url}. Try a different source.'

### Step 2: Defining the tools

In [37]:
tools = []

In [38]:
search_web_function = {
    "name" : "search_web",
    "desription" : "Seach the web using ddgs browser. Return the top 3 results",
    "parameters": {
        "type" : "object",
        "properties": {
            "query" : {
                "type" : "string",
                "description" : "The search query to find relevant websites"
            }
            
        },
        "required" : ['query']
    }
}

In [39]:
tools.append({'type' : 'function', 'function': search_web_function})

In [40]:
fetch_url_function = {
    "name" : "fetch_url",
    "description" : "Fetch and extract the main text content from a web page",
    "parameters" : {
        "type" : "object",
        "properties": {
            "url" : {
                "type" : "string",
                "description" : "The url of the webpage to fetch and extract text from."
            }
        },
        "required" : ['url']
    }
}

In [41]:
tools.append({'type' : 'function', 'function': fetch_url_function})

### Testing

In [42]:
# search_web(query= 'AI Healthcare in 2030')
# fetch_url(url = "https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f")
# # fetch_url(url = "https://www.weforum.org/stories/health-and-healthcare-systems/future-of-artificial-intelligence-healthcare-delivery/")


In [43]:
# tools

### Step 3: Tool call handler

In [52]:
def handle_tool_call(tool_calls:list):
        

        tool_call_results = []

        for tool_call in tool_calls:

            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            #  --- for debugging
            print(f" \U0001F527 the function name is {function_name} with arguments {args}")
               
            if function_name == 'search_web':
                result = search_web(args['query']) 

                content = f"Search results : {result}"

            elif function_name == "fetch_url":
                result = fetch_url(args['url'])
                content = f"Fetched URL content : {result}"

            # elif function_name == "insert_function3_name":

            #     content =insert_function3_name (args['message']})

            else:
                 content = f"Unknown function: {function_name}"

            #package into a dictionary for the llm to see what we got out from the tool call
            tool_call_result = {
            'role' : 'tool',
            'content' : content ,
                'tool_call_id' : tool_call.id   
            }

            print(f"this is what the tool_call_result from handle tool_call looks like: {tool_call_result}")

            #appending each tool_call_result to the tool_call_results list of dictionaries
            tool_call_results.append(tool_call_result)

        return tool_call_results

### Step 4: System Prompt

#### This tells the LLM who it is and how to behave. The key things:
* - What its job is
* - What tools it has
* - What process to follow
* - What output format to produe

In [76]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

IMPORTANT: DO NOT just tool call over and over again. You need to THINK before doing a tool call. This is not a comprehensive brief if you do not think before tool calls. I want to see you working trhough deciding what are the
best sources or urls to use. You NEED AT LEAST 6 sources. Make sure to think through which is the best 6 of all your sources. Try and think between tool calls to determine if you need to search the web more for a better topic search
or fetch a different url. You should use your discernment to try and think through the research brief to create a concise 2 paragraph brief, with cited sources. 

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief
7. You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""



### Step 5: Agentic Loop

In [79]:
#Writing the agentic loop for the research agent:
#max_iterations can be anything
def run_research_agent(topic: str, max_iterations: int = 10):
    """ Run the research agent on a topic and return the research brief 

        Args:
            topic: The topic to search
            max_iterations: safely limit to prevent infinte loops    
    """
    print(f'\n\U0001F50D Starting research on : {topic}')
    print('=' * 60)
    #Initialize converstation message list with system prompt and research task
    messages = [
        {'role' : 'system', 'content': RESEARCH_AGENT_PROMPT},
        {'role' : 'user', 'content': f'Research the following topic and produce a comprehensive brief:\n {topic}'}
    ]

    print(f'MESSAGES: BEFORE WHILE LOOP {messages}')
    #LOOP:
    iteration = 0
    while iteration < max_iterations:
        iteration +=1
        print(f'\n--- Iteration {iteration} ---\n')

        #1. Call the LLM and get response
        response = client.chat.completions.create(
            model = 'gpt-4.1-mini',
            messages=messages,
            tools = tools
        )
        #extracting the message:
        message = response.choices[0].message
        print(f'This is the message before tool call: {message}')
        #2. Check if LLM called tools
        if message.tool_calls:
            messages.append(message)
            print(f"These are the tool calls: {message.tool_calls}")
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
            print(f'This is the MESSAGES during tool call: {message}')

        #3. Otherwise: no tools were called, read message content
        else:
            content = message.content
            #Check if DONE: , then return
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f'\n\u2705 Research Complete!')
                return research_brief
            #Otherwise: not yet done
            #loop continues to next iteration
            else:
                print(f'  \U0001F4AD Agent is Thinking:')
                pprint(content[100:])
                

        #4. If we're entering the final iteration, force a final answer
        if(iteration == max_iterations - 1):
            print("  \u26a0 Safety Limit Reached. Stopping research in next iteration")
            messages.append({"role" : "user", "content": "You have reached the maximum number of iterations. You must deliver your research brief now. You MUST respond with DONE: followed by your brief."})

    #Fallback return
    return "Research incomplete. Maximum iterations reached without finalizing brief."



### Let's Run it!

In [78]:
research_brief = run_research_agent(topic= "AI in healthcare in 2030")
display(Markdown(research_brief))


🔍 Starting research on : AI in healthcare in 2030
MESSAGES: BEFORE WHILE LOOP [{'role': 'system', 'content': 'You are a research specialist. Your job is to research a given topic\nand produce a comprehensive research brief.\n\nIMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. \nONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.\n\nIMPORTANT: DO NOT just tool call over and over again. You need to THINK before doing a tool call. This is not a comprehensive brief if you do not think before tool calls. I want to see you working trhough deciding what are the\nbest sources or urls to use. You NEED AT LEAST 6 sources. Make sure to think through which is the best 6 of all your sources. Try and think between tool calls to determine if you need to search the web more for a better topic search\nor fetch a different url. You should use your discernment to try and think thro

AttributeError: 'list' object has no attribute 'apppend'